# Experiment 2 — Temporal Consistency (T05 refactor)

**Metric:** adjacent-frame cosine similarity of the upsampled features (per pixel, float64), DAVIS-2017 val (30 clips), VideoMAE backbone.
**Conditions:** `raw` (LR VideoMAE features, no upsampling), `bilinear`, `anyup2d` (per frame), `anyup3d`.

Built on the shared modules (`anyup.backbones`, `anyup.video`, `anyup.upsamplers`, `anyup.davis`, `anyup.metrics`); see `docs/plan/P1_experiments/T05_exp2_exp3_refactor.md`.

- **New protocol** (`LEGACY = False`): all frames, VideoMAE chunks of 16 with 8 frames of backbone-only overlap, output 224², AnyUp3D `q_chunk_size=4096`. Temporal cosine is reported for all pairs, pairs within a chunk, and pairs crossing a chunk boundary.
- **`LEGACY = True`** (temporary regression check): first 2 clips, 16 linspace frames, one chunk, AnyUp2D + AnyUp3D, and the old Exp 2 functions re-run inline on the **same backbone chunk**. PASS = upsampled outputs identical (max_abs = 0) and per-clip cosine within `COS_TOL`.

No feature cache: everything is recomputed each run; results go to Drive (`<OUTPUT_ROOT>/<RUN_NAME>/exp2_per_item.csv` + `exp2_summary.json`). T4 is enough. **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-task02"   # T05 continues on the T02 branch
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
LEGACY       = False   # ← True = regression check vs the old Exp 2 code (temporary, removed later)
N_CLIPS      = None    # None = all 30 val clips; int = first N (smoke test). LEGACY forces 2
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # step6000 vs 6500 still open (plan.md)
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T05/exp2"
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0

CONDITIONS   = ["raw", "bilinear", "anyup2d", "anyup3d"]
BASELINES    = ["raw", "bilinear", "anyup2d"]   # paired_stats: anyup3d vs each of these (if run)

GUIDE_SIZE   = 224     # ↓ guide = output size of every upsampler → memory/time of all of them.
                       #   Dependent: Q_CHUNK (reduce it when this grows), GPU memory of one chunk output (8·768·GUIDE_SIZE²·4 B)
CHUNK_SIZE   = 16      # fixed by VideoMAE (NUM_FRAMES=16 position embeddings); 16 frames → 8 feature frames
OVERLAP      = 8       # ↓ backbone-only context frames shared with the previous chunk; cost × 16/(16−OVERLAP).
                       #   Dependent: CHUNK_SIZE (must be < it), VideoMAE temporal_stride=2 (must be a multiple)
Q_CHUNK      = 4096    # ↓ AnyUp3D query chunk → peak GPU memory (T03: 4096 vs None max_abs 3.8e-6 at 112²).
                       #   Dependent: GUIDE_SIZE, CHUNK_SIZE

RUN_VISUALS  = False   # optional PCA figure (Cell 8)
VIZ_CLIP     = None    # None = first clip run
PCA_SAMPLE   = 50_000  # ↓ tokens used to FIT PCA; only fitting speed

COS_TOL      = 1e-4    # legacy: |new float64 − old float32| per-clip temporal cosine
N_LEGACY_FRAMES = 16   # legacy: np.linspace frames (old Exp 2 N_FRAMES)

if LEGACY:
    N_CLIPS    = 2
    CONDITIONS = ["anyup2d", "anyup3d"]     # old Exp 2 conditions
    OVERLAP    = 0                          # 16 frames = exactly one chunk

RUN_NAME = f"{'legacy' if LEGACY else 'new'}_g{GUIDE_SIZE}_c{CHUNK_SIZE}_o{OVERLAP}_q{Q_CHUNK}"
OUT_DIR  = f"{OUTPUT_ROOT}/{RUN_NAME}"
print(f"LEGACY={LEGACY} RUN_NAME={RUN_NAME} CONDITIONS={CONDITIONS}")

In [ ]:
# ── Cell 3 · Imports, data, models ─────────────────────────────────────────
import random, time, math
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from anyup import davis, metrics
from anyup.backbones import BackboneLoader
from anyup.video import TensorSource, iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader

def set_seed(seed):
    """Fixed seeds (the pipeline has no random step; recorded for reproducibility)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

RESULTS = []   # legacy checks: (name, passed: bool | None for INFO, detail)
def record(name, passed, detail):
    RESULTS.append((name, passed, detail))
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")

set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))

ROOT  = davis.download_davis(DAVIS_DEST)
CLIPS = davis.list_clips(ROOT, "val")
if N_CLIPS is not None:
    CLIPS = CLIPS[:N_CLIPS]
print(f"{len(CLIPS)} clips: {CLIPS}")

bb  = BackboneLoader.load("videomae", device=DEVICE, guide_size=GUIDE_SIZE)
UPS = {}
if "bilinear" in CONDITIONS:
    UPS["bilinear"] = UpsamplerLoader.load("bilinear", device=DEVICE)
if "anyup2d" in CONDITIONS:
    UPS["anyup2d"] = UpsamplerLoader.load("anyup2d", device=DEVICE)
if "anyup3d" in CONDITIONS:
    UPS["anyup3d"] = UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE, q_chunk_size=Q_CHUNK)
print("upsamplers:", list(UPS))

In [ ]:
# ── Cell 4 · LEGACY only: old Exp 2 code (verbatim) ────────────────────────
# Old Cell 1 constructors + Cell 4/5/6 functions, copied verbatim. Changes (T05 log):
#   - AnyUp3D state dict from the safe loader (load_anyup3d_checkpoint), not torch.load(weights_only=False)
#   - run_anyup3d passes q_chunk_size=Q_CHUNK (Q17; old code had none → OOM at 224² on T4)
#   - AnyUp2D weights loaded with weights_only=True (safe load, as the T03 wrapper)
if LEGACY:
    from anyup.model import AnyUp
    from anyup.model_2d import AnyUp2D
    from anyup.upsamplers import load_anyup3d_checkpoint
    device = torch.device(DEVICE)

    model_3d = AnyUp(
        input_dim       = 3,
        qk_dim          = 128,
        kernel_size     = 1,
        kernel_size_lfu = 5,
        window_ratio    = 0.1,
        num_heads       = 4,
        t_k             = 3,
        window_t        = 4,
    ).to(device).eval()
    sd = load_anyup3d_checkpoint(CKPT_ANYUP3D)["anyup"]
    missing, unexpected = model_3d.load_state_dict(sd, strict=True)
    print(f"AnyUp3D loaded | missing={missing} | unexpected={unexpected}")

    model_2d = AnyUp2D().to(device).eval()
    sd_2d    = torch.hub.load_state_dict_from_url(
        "https://github.com/wimmerth/anyup/releases/download/checkpoint/anyup_paper.pth",
        map_location=device,
        weights_only=True,                                  # + safe load (T05; old code used the default)
    )
    model_2d.load_state_dict(sd_2d)
    print("AnyUp2D loaded")

    def run_anyup3d(feats, guides):
        feat_vol  = feats.permute(1, 0, 2, 3).unsqueeze(0).to(device)    # (1, 768, 8, 14, 14)
        guide_vol = guides.permute(1, 0, 2, 3).unsqueeze(0).to(device)   # (1, 3, 8, 224, 224)
        with torch.no_grad():
            out = model_3d(guide_vol, feat_vol, q_chunk_size=Q_CHUNK)    # guide first; + q_chunk_size (Q17)
        return out.squeeze(0).permute(1, 0, 2, 3).cpu()                  # (8, 768, 224, 224)

    def run_anyup2d(feats, guides):
        results = []
        for t in range(feats.shape[0]):        # iterate over T=8 frames independently
            feat_t  = feats[t].unsqueeze(0).to(device)    # (1, 768, 14, 14)
            guide_t = guides[t].unsqueeze(0).to(device)   # (1, 3, 224, 224)
            with torch.no_grad():
                out_t = model_2d(guide_t, feat_t)          # guide first, features second
            results.append(out_t.squeeze(0).cpu())         # (768, 224, 224)
        return torch.stack(results, dim=0)                 # (8, 768, 224, 224)

    def adjacent_cosine_sim(feats):
        feats_norm = F.normalize(feats, dim=1)                      # (T, C, H, W)
        sim        = (feats_norm[:-1] * feats_norm[1:]).sum(dim=1)  # (T-1, H, W)
        return sim.mean().item()

    OLD_RUN = {"anyup3d": run_anyup3d, "anyup2d": run_anyup2d}
    # wrapper weights == old reference weights (same file, same keys)
    for name, ref in (("anyup3d", model_3d), ("anyup2d", model_2d)):
        w_sd, r_sd = UPS[name].model.state_dict(), ref.state_dict()
        same = list(w_sd) == list(r_sd) and all(torch.equal(w_sd[k], r_sd[k].to(w_sd[k].device)) for k in w_sd)
        record(f"{name} wrapper weights == old reference weights", same, f"{len(w_sd)} tensors")
else:
    print("LEGACY=False — skipped")

In [ ]:
# ── Cell 5 · Main loop ──────────────────────────────────────────────────────
def clip_source(clip):
    """New protocol: all frames. LEGACY: N_LEGACY_FRAMES linspace frames (old Exp 2 Cell 2)."""
    src = davis.open_clip(ROOT, clip)                                     # frames decoded on demand, [0, 1]
    if not LEGACY:
        return src
    idx = np.linspace(0, len(src) - 1, N_LEGACY_FRAMES, dtype=int).tolist()
    return TensorSource(src.get(idx))                                     # (16, 3, 480, W)

rows, frame_checks, LEGACY_COS = [], [], {}
t_start = time.perf_counter()
for ci, clip in enumerate(CLIPS):
    src = clip_source(clip)
    n_frames = len(src)
    # backbone ONCE per clip; chunks stay on the GPU (≈ 10 MB each at 224²)
    chunks = list(iter_backbone_chunks(bb, src, chunk_size=CHUNK_SIZE, overlap=OVERLAP))
    fidx = [f for ch in chunks for f in ch.meta["frame_index"]]          # source frame per feature frame, in order
    frame_checks.append(fidx == list(range(0, n_frames, bb.temporal_stride)))
    n_tokens = len(fidx)

    for cond in CONDITIONS:
        t0 = time.perf_counter()
        stream = metrics.TemporalCosineStream()
        for ch in chunks:
            # (t, 768, 14, 14) raw | (t, 768, GUIDE_SIZE, GUIDE_SIZE) upsampled, on the GPU
            feats = ch.features if cond == "raw" else UPS[cond](ch).features
            stream.update(feats, new_chunk=ch.meta["chunk_id"] > 0)
            if LEGACY and cond in OLD_RUN:
                old = OLD_RUN[cond](ch.features, ch.guide)                   # same chunk → old function, CPU
                new = feats.cpu()
                record(f"{clip} {cond}: new == old output", torch.equal(new, old),
                       f"shape={tuple(new.shape)} max_abs={(new - old).abs().max().item():.3e}")
                LEGACY_COS[(clip, cond)] = adjacent_cosine_sim(old)
                del old, new
            del feats
        r = stream.result()
        rows.append(dict(clip=clip, condition=cond, all=r["all"], within=r["within"], cross=r["cross"],
                         n_pairs=r["n_pairs"], n_cross=r["n_cross"], n_frames=n_frames, n_tokens=n_tokens,
                         n_chunks=len(chunks), time_s=round(time.perf_counter() - t0, 2)))
        fmt = lambda v: "—" if v is None else f"{v:.4f}"
        print(f"[{ci + 1}/{len(CLIPS)}] {clip:<20} {cond:<9} all={r['all']:.4f} within={fmt(r['within'])} "
              f"cross={fmt(r['cross'])} frames={n_frames} tokens={n_tokens} chunks={len(chunks)} "
              f"{time.perf_counter() - t0:.1f}s")
    del chunks
    torch.cuda.empty_cache()
print(f"done in {time.perf_counter() - t_start:.0f}s | frame_index == range(0, N, 2) for every clip: {all(frame_checks)}")

In [ ]:
# ── Cell 6 · Summary, paired statistics, save ──────────────────────────────
summary = {"per_condition": {}, "paired_anyup3d_vs": {}, "frame_index_ok": all(frame_checks)}
print(f"{'condition':<10} {'all':>16} {'within':>16} {'cross':>16}")
for cond in CONDITIONS:
    rs = [r for r in rows if r["condition"] == cond]
    s = {}
    for key in ("all", "within", "cross"):
        v = [r[key] for r in rs if r[key] is not None]
        s[key] = {"mean": float(np.mean(v)) if v else None, "std": float(np.std(v)) if v else None, "n": len(v)}
    summary["per_condition"][cond] = s
    fmt = lambda d: "—" if d["mean"] is None else f"{d['mean']:.4f}±{d['std']:.4f}"
    print(f"{cond:<10} {fmt(s['all']):>16} {fmt(s['within']):>16} {fmt(s['cross']):>16}")

if "anyup3d" in CONDITIONS:
    ours = {r["clip"]: r["all"] for r in rows if r["condition"] == "anyup3d"}
    for base in BASELINES:
        if base not in CONDITIONS:
            continue
        b = {r["clip"]: r["all"] for r in rows if r["condition"] == base}
        st = metrics.paired_stats([ours[c] for c in CLIPS], [b[c] for c in CLIPS])
        summary["paired_anyup3d_vs"][base] = st
        fp = lambda p: "—" if p is None else f"{p:.2e}"
        print(f"anyup3d vs {base:<9}: Δ={st['mean_delta']:+.4f} improved {st['n_improved']}/{st['n']} "
              f"sign p2={fp(st['sign_p_two_sided'])} wilcoxon p2={fp(st['wilcoxon_p_two_sided'])}")

config = dict(LEGACY=LEGACY, N_CLIPS=N_CLIPS, clips=CLIPS, CKPT_ANYUP3D=CKPT_ANYUP3D, SEED=SEED,
              CONDITIONS=CONDITIONS, GUIDE_SIZE=GUIDE_SIZE, CHUNK_SIZE=CHUNK_SIZE, OVERLAP=OVERLAP,
              Q_CHUNK=Q_CHUNK, backbone="videomae", RUN_NAME=RUN_NAME,
              anyup3d_meta={k: v for k, v in UPS["anyup3d"]._extra_meta().items()} if "anyup3d" in UPS else None,
              anyup2d_meta=UPS["anyup2d"]._extra_meta() if "anyup2d" in UPS else None,
              repo_branch=REPO_BRANCH)
if LEGACY:
    summary["legacy_checks"] = [dict(name=n, passed=p, detail=d) for n, p, d in RESULTS]
csv_path, json_path = metrics.save_results(OUT_DIR, rows, summary, config, name="exp2")
print("saved:", csv_path, json_path)

In [ ]:
# ── Cell 7 · LEGACY only: temporal cosine new (float64) vs old (float32) ───
if LEGACY:
    for (clip, cond), old_v in LEGACY_COS.items():
        new_v = next(r["all"] for r in rows if r["clip"] == clip and r["condition"] == cond)
        d = abs(new_v - old_v)
        record(f"{clip} {cond}: temporal cosine new vs old", d < COS_TOL,
               f"new={new_v:.6f} old={old_v:.6f} |diff|={d:.2e} (tol {COS_TOL:.0e})")
    n_fail = sum(p is False for _, p, _ in RESULTS)
    print(f"\nLEGACY SUMMARY: {len(RESULTS)} checks, {n_fail} failed")
    for n, p, d in RESULTS:
        print(f"  [{'INFO' if p is None else ('PASS' if p else 'FAIL')}] {n}: {d}")
    summary["legacy_checks"] = [dict(name=n, passed=p, detail=d) for n, p, d in RESULTS]
    metrics.save_results(OUT_DIR, rows, summary, config, name="exp2")   # re-save with the cosine checks
else:
    print("LEGACY=False — skipped")

In [ ]:
# ── Cell 8 · Optional: PCA figure (first chunk of VIZ_CLIP, one row per upsampler) ─
if RUN_VISUALS:
    from sklearn.decomposition import PCA
    clip = VIZ_CLIP or CLIPS[0]
    ch = next(iter_backbone_chunks(bb, clip_source(clip), chunk_size=CHUNK_SIZE, overlap=OVERLAP))
    conds = [c for c in CONDITIONS if c != "raw"]
    outs = {c: UPS[c](ch).features.cpu() for c in conds}                  # each (8, 768, G, G)
    T, C, H, W = outs[conds[0]].shape
    flat = np.concatenate([o.permute(0, 2, 3, 1).reshape(-1, C).numpy() for o in outs.values()], axis=0)
    idx = np.random.default_rng(SEED).choice(len(flat), min(PCA_SAMPLE, len(flat)), replace=False)
    pca = PCA(n_components=3).fit(flat[idx])
    del flat

    def apply_pca(feats):
        proj = pca.transform(feats.permute(0, 2, 3, 1).reshape(-1, C).numpy())
        proj = (proj - proj.min()) / (proj.max() - proj.min() + 1e-8)
        return proj.reshape(T, H, W, 3)

    fig, axes = plt.subplots(len(conds), T, figsize=(3 * T + 1.5, 3 * len(conds)), squeeze=False)
    for r, c in enumerate(conds):
        imgs = apply_pca(outs[c])
        for t in range(T):
            axes[r, t].imshow(imgs[t]); axes[r, t].axis("off")
            if r == 0:
                axes[r, t].set_title(f"frame {ch.meta['frame_index'][t]}", fontsize=9)
        fig.text(0.01, 1 - (r + 0.5) / len(conds), c, va="center", ha="left", fontsize=10, fontweight="bold", rotation=90)
    fig.suptitle(f"PCA of upsampled features — {clip}", fontsize=12, y=1.02)
    plt.tight_layout(rect=[0.04, 0, 1, 1])
    viz_path = f"{OUT_DIR}/pca_{clip}.png"
    plt.savefig(viz_path, dpi=150, bbox_inches="tight"); plt.show()
    print("saved:", viz_path)
else:
    print("RUN_VISUALS=False — skipped")